# DAU Drop Analysis — Mar 2026

Investigating the acceleration of DAU decline observed in early 2026. Analysis covers:
1. Data pull and feature engineering
2. Timeline overview with installs and retention overlays
3. Cohort attribution — quantifying which install cohorts are driving the drop

In [234]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html


## 1. Get data

Query DSI activity for the last 400 days (lookback extended to reach the Feb 2025 YoY baseline window). Data is cached to a pickle to avoid re-running the query on every kernel restart.

In [235]:
# hide-output
query_location = './sql/dsi_activity.sql'
parameters = {
    'lookback_days': 440,  # needs to reach Feb 8 2025 for the "same time last year" baseline
}

bqc = BigQueryConnector()

cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 222.64 GB when run.
Estimated query cost: $1.49


In [236]:
# hide-output
import os

force_refresh = True  # Set to True to force re-querying the data and updating the pickle file

if not os.path.exists('./data/dsi_activity.pkl') or force_refresh:
    data = bqc.get(query=query_location, is_path=True, query_parameters=parameters)
    data.to_pickle('./data/dsi_activity.pkl')
else:
    data = pd.read_pickle('./data/dsi_activity.pkl')
    print("Pickle already exists, skipping query.")

In [237]:
# hide-output
data

,dt,install_dt,install_wk,days_since_install,user_id
0,2025-01-14,2021-06-25,2021-06-20,1299,DB8C7D10E0ED4982
1,2025-01-14,2021-06-25,2021-06-20,1299,ED0A3D30A8FEA66A
2,2025-01-14,2021-06-25,2021-06-20,1299,592661DDA35148C8
3,2025-01-14,2021-06-27,2021-06-27,1297,87B93303CAC58EDB
4,2025-01-14,2021-06-28,2021-06-27,1296,6C00106B2AC354F8
...,...,...,...,...,...
61524377,2026-03-29,2026-03-29,2026-03-29,0,623C4E203B2D8336
61524378,2026-03-29,2026-03-29,2026-03-29,0,4F2C9FD127CFAFBC
61524379,2026-03-29,2026-03-29,2026-03-29,0,42023115822D9096
61524380,2026-03-29,2026-03-29,2026-03-29,0,8D13C257D13938ED


### Player level data pull

In [238]:
# hide-output
query_location = './sql/player_level.sql'
parameters = {
    'lookback_days': 400,  # needs to reach Feb 8 2025 for the "same time last year" baseline
}

bqc = BigQueryConnector()

cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 109.03 GB when run.
Estimated query cost: $0.73


In [239]:
# hide-output
import os

force_refresh = True  # Set to True to force re-querying the data and updating the pickle file

if not os.path.exists('./data/player_level.pkl') or force_refresh:
    pl_data = bqc.get(query=query_location, is_path=True, query_parameters=parameters)
    pl_data.to_pickle('./data/player_level.pkl')
else:
    pl_data = pd.read_pickle('./data/player_level.pkl')
    print("Pickle already exists, skipping query.")

In [240]:
# hide-output
pl_data

,dt,max_level,platform,user_id,active_users
0,2025-02-23,1,IOS,C363CF1DF9B365B8,1
1,2025-02-23,1,AND,2810941BF173CC9D,1
2,2025-02-23,1,AND,4A529ADF2C5D19AD,1
3,2025-02-23,1,AND,E91616793A953867,1
4,2025-02-23,1,IOS,4A5AB91090DC2822,1
...,...,...,...,...,...
54040033,2026-03-29,200,AND,D8B44BF208F8A3CD,1
54040034,2026-03-29,200,AND,41734C6BAFD57075,1
54040035,2026-03-29,200,AND,36C0B9765EDE468C,1
54040036,2026-03-29,200,AND,7AABEF719EB7EE57,1


## 2. Feature Engineering

- **`days_since_install_bin`**: fixed-width age buckets (0–100d, 100–365d, 365–730d, 730d+)
- **`install_dt_bin`**: install cohort label — yearly for 2021–2024, quarterly for 2025, weekly for 2026

In [241]:
# hide-output
# Create fixed-width bins
data['days_since_install_bin'] = pd.cut(data['days_since_install'], bins=[0, 100, 365, 730, float('inf')], labels=['A.0-100 days', 'B.100-365 days', 'C.365-730 days', 'D.730+ days'])

# Or use qcut for equal-frequency bins
# data['days_since_install_bin'] = pd.qcut(data['days_since_install'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

data.head()

,dt,install_dt,install_wk,days_since_install,user_id,days_since_install_bin
0,2025-01-14,2021-06-25,2021-06-20,1299,DB8C7D10E0ED4982,D.730+ days
1,2025-01-14,2021-06-25,2021-06-20,1299,ED0A3D30A8FEA66A,D.730+ days
2,2025-01-14,2021-06-25,2021-06-20,1299,592661DDA35148C8,D.730+ days
3,2025-01-14,2021-06-27,2021-06-27,1297,87B93303CAC58EDB,D.730+ days
4,2025-01-14,2021-06-28,2021-06-27,1296,6C00106B2AC354F8,D.730+ days


In [242]:
# hide-output
# Bin install_dt by year and quarter/week
def bin_install_dt(dt):
    year = dt.year
    if year < 2021:
        return 'Before 2021'
    elif year in [2021, 2022, 2023, 2024]:
        quarter = (dt.month - 1) // 3 + 1
        return f'{year}'
    elif year in [2025, 2026]:
        quarter = (dt.month - 1) // 3 + 1
        return f'{year} Q{quarter}'
    elif year == 2049: # Just a trick not to use this bin variation
        week = dt.isocalendar().week
        return f'2049 W{week}'
    else:
        return 'After 2049'
data['install_dt_bin'] = data['install_wk'].apply(bin_install_dt)
data.head()

,dt,install_dt,install_wk,days_since_install,user_id,days_since_install_bin,install_dt_bin
0,2025-01-14,2021-06-25,2021-06-20,1299,DB8C7D10E0ED4982,D.730+ days,2021
1,2025-01-14,2021-06-25,2021-06-20,1299,ED0A3D30A8FEA66A,D.730+ days,2021
2,2025-01-14,2021-06-25,2021-06-20,1299,592661DDA35148C8,D.730+ days,2021
3,2025-01-14,2021-06-27,2021-06-27,1297,87B93303CAC58EDB,D.730+ days,2021
4,2025-01-14,2021-06-28,2021-06-27,1296,6C00106B2AC354F8,D.730+ days,2021


## 3. Aggregations

### 3.1 Daily active users by install cohort

In [271]:
# hide-output
# --- Period definitions (derived from raw data, no circular dependency) ---
recent_end   = pd.Timestamp(data['dt'].max())
recent_start = recent_end - pd.Timedelta(days=20)
dow          = recent_start.dayofweek

b1_anchor = pd.Timestamp('2025-04-21')
b1_start  = b1_anchor + pd.Timedelta(days=(dow - b1_anchor.dayofweek) % 7)
b1_end    = b1_start + pd.Timedelta(days=20)

b2_anchor = pd.Timestamp('2025-10-20')
b2_start  = b2_anchor + pd.Timedelta(days=(dow - b2_anchor.dayofweek) % 7)
b2_end    = b2_start + pd.Timedelta(days=20)

print(f"Recent:     {recent_start.date()} → {recent_end.date()}")
print(f"Baseline 1: {b1_start.date()} → {b1_end.date()}")
print(f"Baseline 2: {b2_start.date()} → {b2_end.date()}")

# --- Uncapped: all tenures included (used in section 4 timeline charts) ---
cohort_data = data.groupby(['dt', 'install_dt_bin'], observed=True).agg(
    users=('user_id', 'nunique')
).reset_index()

cohort_data['total_users'] = cohort_data.groupby('dt')['users'].transform('sum')
cohort_data['pct_users'] = cohort_data['users'] / cohort_data['total_users']

print("\nUncapped — all tenures included")
cohort_data

Recent:     2026-03-09 → 2026-03-29
Baseline 1: 2025-04-21 → 2025-05-11
Baseline 2: 2025-10-20 → 2025-11-09

Uncapped — all tenures included


,dt,install_dt_bin,users,total_users,pct_users
0,2025-01-14,2021,8231,185679,0.044329
1,2025-01-14,2022,29335,185679,0.157988
2,2025-01-14,2023,39287,185679,0.211586
3,2025-01-14,2024,95337,185679,0.513451
4,2025-01-14,2025 Q1,13489,185679,0.072647
...,...,...,...,...,...
3081,2026-03-29,2025 Q1,5152,95799,0.053779
3082,2026-03-29,2025 Q2,4637,95799,0.048403
3083,2026-03-29,2025 Q3,5619,95799,0.058654
3084,2026-03-29,2025 Q4,8820,95799,0.092068


### 3.2 New installs (dt = install_dt)

In [244]:
# hide-output
new_installs_by_dt = (
    data[data['days_since_install'] == 0]
    .groupby('dt')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'new_installs'})
)
new_installs_by_dt['dt'] = pd.to_datetime(new_installs_by_dt['dt'])
new_installs_by_dt

,dt,new_installs
0,2025-01-14,4270
1,2025-01-15,4312
2,2025-01-16,4896
3,2025-01-17,6390
4,2025-01-18,6422
...,...,...
435,2026-03-25,1790
436,2026-03-26,1546
437,2026-03-27,1549
438,2026-03-28,1583


### 3.3 Retention — D1, D7, D28

Keyed to install date: for each day's cohort, measures what % were still active 1, 7, and 28 days later. The last 1/7/28 days of data will be NaN as the retention window hasn't closed yet.

In [245]:
# hide-output
# Retention D1 / D7 / D28 — keyed to install_dt (x-axis = day cohort was acquired)
data['dt'] = pd.to_datetime(data['dt'])

cohort_size = (
    data[data['days_since_install'] == 0]
    .groupby('dt')['user_id']
    .nunique()
)

retention_rows = {}
for d in [1, 7, 28]:
    retained = (
        data[data['days_since_install'] == d]
        .groupby('dt')['user_id']
        .nunique()
    )
    # shift activity date back to install date
    retained.index = pd.to_datetime(retained.index) - pd.Timedelta(days=d)
    retention_rows[f'D{d}'] = (retained / cohort_size * 100)

retention_df = pd.DataFrame(retention_rows).reset_index().rename(columns={'index': 'dt'})
retention_df

,dt,D1,D7,D28
0,2024-12-17,NaN,NaN,NaN
1,2024-12-18,NaN,NaN,NaN
2,2024-12-19,NaN,NaN,NaN
3,2024-12-20,NaN,NaN,NaN
4,2024-12-21,NaN,NaN,NaN
...,...,...,...,...
463,2026-03-25,32.793296,NaN,NaN
464,2026-03-26,32.858991,NaN,NaN
465,2026-03-27,34.861201,NaN,NaN
466,2026-03-28,34.301958,NaN,NaN


## 4. Timeline Charts

Full date-range view of DAU by install cohort, overlaid with:
- **New installs** (black, left-shifted right axis) and **avg daily installs per cohort period** (green dashed)
- **D1 / D7 / D28 retention %** (dotted, second right axis, 0–100%)
- Red reference lines marking the baseline and recent analysis windows

In [246]:
fig = px.line(
            cohort_data, 
            x='dt', 
            y='users', 
            color='install_dt_bin', 
            title='User Cohorts by Install Week and Days Since Install',
            labels={'dt': 'Date', 'users': 'Number of Users'},
            hover_data={'install_dt_bin': True, 'users': ':.0f'},)

fig.update_layout(xaxis_title='Date', 
                  yaxis_title='Number of Users', 
                  legend_title='Install Cohort',
                  height=800,
                  width=1500)

# --- New installs (right axis y2) ---
fig.add_trace(go.Scatter(
    x=new_installs_by_dt['dt'],
    y=new_installs_by_dt['new_installs'],
    mode='lines',
    name='New Installs',
    line=dict(color='black', width=2),
    yaxis='y2',
))

# --- Retention D1 / D7 / D28 (right axis y3) ---
retention_colors = {'D1': 'gold', 'D7': 'darkorange', 'D28': 'saddlebrown'}
for col, color in retention_colors.items():
    fig.add_trace(go.Scatter(
        x=retention_df['dt'],
        y=retention_df[col],
        mode='lines',
        name=f'{col} Retention %',
        line=dict(color=color, width=1.5, dash='solid'),
        yaxis='y3',
    ))

# --- Avg daily installs per install_dt_bin period (step line, y2) ---
new_installs_by_dt['install_dt_bin'] = new_installs_by_dt['dt'].apply(bin_install_dt)
avg_by_period = (
    new_installs_by_dt
    .groupby('install_dt_bin', observed=True)['new_installs']
    .mean()
    .rename('avg_new_installs')
)
avg_step = (
    new_installs_by_dt[['dt', 'install_dt_bin']]
    .merge(avg_by_period, on='install_dt_bin')
    .sort_values('dt')
)
fig.add_trace(go.Scatter(
    x=avg_step['dt'],
    y=avg_step['avg_new_installs'],
    mode='lines',
    name='Avg Daily Installs (per period)',
    line=dict(color='green', width=2, dash='dash'),
    yaxis='y2',
))

fig.update_layout(
    xaxis=dict(domain=[0, 0.95]),
    yaxis=dict(title='Active Users'),
    yaxis2=dict(
        title='New Installs',
        overlaying='y',
        side='right',
        anchor='free',
        position=0.83,
        showgrid=False,
    ),
    yaxis3=dict(
        title='Retention %',
        overlaying='y',
        side='right',
        anchor='free',
        position=1.0,
        range=[0, 100],
        showgrid=False,
        ticksuffix='%',
    ),
    legend=dict(
        orientation='v',
        yanchor='middle', y=0.5,
        xanchor='left', x=1.08,
    ),
    margin=dict(r=200),
)

ref_lines = [
    (b1_start.strftime('%Y-%m-%d'), 'BSE1 start'),
    (b1_end.strftime('%Y-%m-%d'),   'BSE1 end'),
    (b2_start.strftime('%Y-%m-%d'), 'BSE2 start'),
    (b2_end.strftime('%Y-%m-%d'),   'BSE2 end'),
    (recent_start.strftime('%Y-%m-%d'), 'P start'),
    (recent_end.strftime('%Y-%m-%d'),   'P end'),
]

for date_str, label in ref_lines:
    fig.add_vline(
        x=pd.Timestamp(date_str).timestamp() * 1000,
        line_width=1,
        line_color='red',
        annotation_text=label,
        annotation_position='top right',
        annotation_font_size=10,
    )

fig.show()

In [247]:
# Create a chart with install cohorts from 2025 onwards, aligned by days since cohort start
cohort_data_2025_plus = cohort_data[cohort_data['install_dt_bin'].isin(['2025 Q1', '2025 Q2', '2025 Q3', '2025 Q4', '2026 Q1'])]

# Get the install start date for each cohort
cohort_starts = {
    '2025 Q1': pd.Timestamp('2025-01-01'),
    '2025 Q2': pd.Timestamp('2025-04-01'),
    '2025 Q3': pd.Timestamp('2025-07-01'),
    '2025 Q4': pd.Timestamp('2025-10-01'),
    '2026 Q1': pd.Timestamp('2026-01-01'),
}

# Add days since cohort start
cohort_data_2025_plus_aligned = cohort_data_2025_plus.copy()
cohort_data_2025_plus_aligned['days_since_cohort_start'] = cohort_data_2025_plus_aligned.apply(
    lambda row: (pd.Timestamp(row['dt']) - cohort_starts[row['install_dt_bin']]).days,
    axis=1
)

fig = px.line(
    cohort_data_2025_plus_aligned,
    x='days_since_cohort_start',
    y='users',
    color='install_dt_bin',
    title='DAU by Install Cohort (2025+) — Aligned by Days Since Cohort Start',
    hover_data={'install_dt_bin': True, 'pct_users': ':.2%', 'users': ':.0f'},
    markers=False,
)

fig.update_layout(
    height=600,
    width=1200,
)

fig.show()

In [248]:
# Create a chart with install cohorts from 2025 onwards, aligned by days since cohort start
cohort_data_2025_plus = cohort_data[cohort_data['install_dt_bin'].isin(['2025 Q1', '2025 Q2', '2025 Q3', '2025 Q4', '2026 Q1'])]

# Get the install start date for each cohort
cohort_starts = {
    '2025 Q1': pd.Timestamp('2025-01-01'),
    '2025 Q2': pd.Timestamp('2025-04-01'),
    '2025 Q3': pd.Timestamp('2025-07-01'),
    '2025 Q4': pd.Timestamp('2025-10-01'),
    '2026 Q1': pd.Timestamp('2026-01-01'),
}

# Add days since cohort start
cohort_data_2025_plus_aligned = cohort_data_2025_plus.copy()
cohort_data_2025_plus_aligned['days_since_cohort_start'] = cohort_data_2025_plus_aligned.apply(
    lambda row: (pd.Timestamp(row['dt']) - cohort_starts[row['install_dt_bin']]).days,
    axis=1
)

fig = px.line(
    cohort_data_2025_plus_aligned,
    x='days_since_cohort_start',
    y='pct_users',
    color='install_dt_bin',
    title='DAU by Install Cohort (2025+) — Aligned by Days Since Cohort Start',
    hover_data={'install_dt_bin': True, 'pct_users': ':.2%', 'users': ':.0f'},
    markers=False,
)

fig.update_layout(
    height=600,
    width=1200,
)

fig.show()

In [249]:
fig = px.area(
            cohort_data, 
            x='dt', 
            y='users', 
            color='install_dt_bin', 
            title='User Cohorts by Install Year/Quarter and Days Since Install',
            labels={'dt': 'Date', 'users': 'Number of Users'},
            hover_data={'install_dt_bin': True, 'users': ':.0f','total_users': ':.0f'},)

fig.update_layout(xaxis_title='Date', 
                  yaxis_title='Number of Users', 
                  legend_title='Install Cohort',
                  height=800,
                  width=1500)

# --- New installs (right axis y2) ---
fig.add_trace(go.Scatter(
    x=new_installs_by_dt['dt'],
    y=new_installs_by_dt['new_installs'],
    mode='lines',
    name='New Installs',
    line=dict(color='black', width=2),
    yaxis='y2',
))

# --- Avg daily installs per install_dt_bin period (step line, y2) ---
new_installs_by_dt['install_dt_bin'] = new_installs_by_dt['dt'].apply(bin_install_dt)
avg_by_period = (
    new_installs_by_dt
    .groupby('install_dt_bin', observed=True)['new_installs']
    .mean()
    .rename('avg_new_installs')
)
avg_step = (
    new_installs_by_dt[['dt', 'install_dt_bin']]
    .merge(avg_by_period, on='install_dt_bin')
    .sort_values('dt')
)
fig.add_trace(go.Scatter(
    x=avg_step['dt'],
    y=avg_step['avg_new_installs'],
    mode='lines',
    name='Avg Daily Installs (per period)',
    line=dict(color='green', width=2, dash='dash'),
    yaxis='y2',
))

# --- Retention D1 / D7 / D28 (right axis y3) ---
retention_colors = {'D1': 'gold', 'D7': 'darkorange', 'D28': 'saddlebrown'}
for col, color in retention_colors.items():
    fig.add_trace(go.Scatter(
        x=retention_df['dt'],
        y=retention_df[col],
        mode='lines',
        name=f'{col} Retention %',
        line=dict(color=color, width=1.5, dash='solid'),
        yaxis='y3',
    ))

fig.update_layout(
    xaxis=dict(domain=[0, 0.95]),
    yaxis=dict(title='Active Users'),
    yaxis2=dict(
        title='New Installs',
        overlaying='y',
        side='right',
        anchor='free',
        position=0.98,
        showgrid=False,
    ),
    yaxis3=dict(
        title='Retention %',
        overlaying='y',
        side='right',
        anchor='free',
        position=1.0,
        range=[0, 100],
        showgrid=False,
        ticksuffix='%',
    ),
    legend=dict(
        orientation='v',
        yanchor='middle', y=0.5,
        xanchor='left', x=1.08,
    ),
    margin=dict(r=200),
)

ref_lines = [
    (b1_start.strftime('%Y-%m-%d'), 'BSE1 start'),
    (b1_end.strftime('%Y-%m-%d'),   'BSE1 end'),
    (b2_start.strftime('%Y-%m-%d'), 'BSE2 start'),
    (b2_end.strftime('%Y-%m-%d'),   'BSE2 end'),
    (recent_start.strftime('%Y-%m-%d'), 'P start'),
    (recent_end.strftime('%Y-%m-%d'),   'P end'),
]

for date_str, label in ref_lines:
    fig.add_vline(
        x=pd.Timestamp(date_str).timestamp() * 1000,
        line_width=1,
        line_color='red',
        annotation_text=label,
        annotation_position='top right',
        annotation_font_size=10,
    )

fig.show()

## 5. DAU Drop Attribution by Cohort

Compare the most recent 21-day window (ending on the latest date available in the data) against two baselines anchored in Apr 2025 and Oct 2025. All windows are 21 days and share the same day-of-week alignment — baseline start dates are shifted forward from their anchors (Apr 21 and Oct 20) to land on the same weekday as the recent period start.

| Period | Anchor | Start | End |
|---|---|---|---|
| Recent | latest data | `recent_end - 20d` | `cohort_data['dt'].max()` |
| Baseline 1 | Apr 21, 2025 | nearest same weekday ≥ Apr 21 | start + 20d |
| Baseline 2 | Oct 20, 2025 | nearest same weekday ≥ Oct 20 | start + 20d |

Each comparison shows: cohort attribution table, absolute delta bar, % share of drop bar, and a day-aligned DAU overlay with regression lines.

In [260]:
# hide-output
# --- Capped: tenure capped to the max days_since_install observed in the baseline 1 window ---
# This ensures the recent period is compared against users at the same lifecycle stage,
# avoiding inflation from long-tenured players who existed in the baseline but not in newer cohorts.
# cohort_data (uncapped) is kept intact for the section 4 timeline charts.
max_tenure_b1 = data[data['dt'].between(b1_start, b1_end)]['days_since_install'].max()
print(f"Capped at {max_tenure_b1} days (max tenure observed in baseline 1 window)")

cohort_data_capped = (
    data[data['days_since_install'] <= max_tenure_b1]
    .groupby(['dt', 'install_dt_bin'], observed=True)
    .agg(users=('user_id', 'nunique'))
    .reset_index()
)
cohort_data_capped['total_users'] = cohort_data_capped.groupby('dt')['users'].transform('sum')
cohort_data_capped['pct_users'] = cohort_data_capped['users'] / cohort_data_capped['total_users']
cohort_data_capped

Capped at 1418 days (max tenure observed in baseline 1 window)


,dt,install_dt_bin,users,total_users,pct_users
0,2025-01-14,2021,8231,185679,0.044329
1,2025-01-14,2022,29335,185679,0.157988
2,2025-01-14,2023,39287,185679,0.211586
3,2025-01-14,2024,95337,185679,0.513451
4,2025-01-14,2025 Q1,13489,185679,0.072647
...,...,...,...,...,...
2951,2026-03-29,2025 Q1,5152,87205,0.059079
2952,2026-03-29,2025 Q2,4637,87205,0.053174
2953,2026-03-29,2025 Q3,5619,87205,0.064434
2954,2026-03-29,2025 Q4,8820,87205,0.101141


In [285]:
import numpy as np

cohort_data_capped['dt'] = pd.to_datetime(cohort_data_capped['dt'])
total_by_dt_capped = cohort_data_capped.groupby('dt', observed=True)['users'].sum().reset_index()

def analyze_dau_drop(cohort_data, total_by_dt, baseline_start, baseline_end, recent_start, recent_end, label):
    """Run cohort attribution and period overlay charts for one baseline vs recent comparison."""

    # --- Cohort attribution ---
    baseline = (
        cohort_data[cohort_data['dt'].between(baseline_start, baseline_end)]
        .groupby('install_dt_bin', observed=True)['users']
        .mean()
        .rename('baseline_avg_dau')
    )
    recent = (
        cohort_data[cohort_data['dt'].between(recent_start, recent_end)]
        .groupby('install_dt_bin', observed=True)['users']
        .mean()
        .rename('recent_avg_dau')
    )

    cohort_drop = pd.concat([baseline, recent], axis=1).fillna(0)
    cohort_drop['delta'] = cohort_drop['recent_avg_dau'] - cohort_drop['baseline_avg_dau']
    cohort_drop['pct_change'] = cohort_drop.apply(
        lambda r: (r['delta'] / r['baseline_avg_dau'] * 100) if r['baseline_avg_dau'] != 0 else float('nan'),
        axis=1,
    )
    total_drop = cohort_drop['delta'].sum()
    cohort_drop['contribution_pct'] = cohort_drop['delta'] / total_drop * 100
    cohort_drop = cohort_drop.sort_values('delta')

    #display(cohort_drop.round(1))

    # --- Overlay: total DAU aligned by day index, dual y-axis, with regression lines ---
    baseline_daily = total_by_dt[total_by_dt['dt'].between(baseline_start, baseline_end)].sort_values('dt').reset_index(drop=True)
    recent_daily   = total_by_dt[total_by_dt['dt'].between(recent_start, recent_end)].sort_values('dt').reset_index(drop=True)
    baseline_daily['day'] = range(len(baseline_daily))
    recent_daily['day']   = range(len(recent_daily))
    baseline_daily['date_str'] = baseline_daily['dt'].dt.strftime('%Y-%m-%d')
    baseline_daily['dow']      = baseline_daily['dt'].dt.strftime('%A')
    recent_daily['date_str']   = recent_daily['dt'].dt.strftime('%Y-%m-%d')
    recent_daily['dow']        = recent_daily['dt'].dt.strftime('%A')

    def regression_line(df):
        x, y = df['day'].values, df['users'].values
        m, b = np.polyfit(x, y, 1)
        return m * x + b

    baseline_trend = regression_line(baseline_daily)
    recent_trend   = regression_line(recent_daily)

    # --- Matched y-axis ranges: span = max of each series' own fluctuation range ---
    baseline_fluct = baseline_daily['users'].max() - baseline_daily['users'].min()
    recent_fluct   = recent_daily['users'].max()   - recent_daily['users'].min()
    y_span = max(baseline_fluct, recent_fluct) * 1.4  # shared span with 40% padding
    baseline_center = baseline_daily['users'].mean()
    recent_center   = recent_daily['users'].mean()
    baseline_yrange = [baseline_center - y_span / 2, baseline_center + y_span / 2]
    recent_yrange   = [recent_center   - y_span / 2, recent_center   + y_span / 2]

    hover_tpl = (
        'Day %{x}<br>'
        'Date: %{customdata[0]}<br>'
        '%{customdata[1]}<br>'
        'DAU: %{y:,.0f}'
        '<extra></extra>'
    )

    fig3 = go.Figure()

    fig3.add_trace(go.Scatter(
        x=baseline_daily['day'], y=baseline_daily['users'],
        mode='lines+markers', name=f'Baseline ({baseline_start} → {baseline_end})',
        line=dict(color='steelblue'), yaxis='y1',
        customdata=baseline_daily[['date_str', 'dow']].values,
        hovertemplate=hover_tpl,
    ))

    fig3.add_trace(go.Scatter(
        x=baseline_daily['day'], y=baseline_trend,
        mode='lines', name='Baseline trend',
        line=dict(color='steelblue', dash='dash', width=2), yaxis='y1',
        hoverinfo='skip',
    ))

    fig3.add_trace(go.Scatter(
        x=recent_daily['day'], y=recent_daily['users'],
        mode='lines+markers', name=f'Recent ({recent_start} → {recent_end})',
        line=dict(color='crimson'), yaxis='y2',
        customdata=recent_daily[['date_str', 'dow']].values,
        hovertemplate=hover_tpl,
    ))

    fig3.add_trace(go.Scatter(
        x=recent_daily['day'], y=recent_trend,
        mode='lines', name='Recent trend',
        line=dict(color='crimson', dash='dash', width=2), yaxis='y2',
        hoverinfo='skip',
    ))

    fig3.update_layout(
        title=f"[{label}] Total DAU — baseline vs recent (aligned by day in period)",
        xaxis_title='Day in period (0 = first day)',
        yaxis=dict(title='DAU — Baseline', color='steelblue', range=baseline_yrange),
        yaxis2=dict(title='DAU — Recent', color='crimson', overlaying='y', side='right', range=recent_yrange),
        height=500, width=1100,
        legend=dict(orientation='h', yanchor='bottom', y=-0.25, xanchor='center', x=0.5),
    )
    fig3.show()

In [286]:

analyze_dau_drop(
    cohort_data=cohort_data_capped,
    total_by_dt=total_by_dt_capped,
    baseline_start=b1_start.strftime('%Y-%m-%d'),
    baseline_end=b1_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b1_start.strftime("%b %d %Y")} Baseline 1',
)

In [287]:
# Baseline 2 anchor: week of Oct 20 2025 — shift to match recent_start's day of week
b2_anchor = pd.Timestamp('2025-10-20')
b2_start = b2_anchor + pd.Timedelta(days=(dow - b2_anchor.dayofweek) % 7)
b2_end = b2_start + pd.Timedelta(days=20)

analyze_dau_drop(
    cohort_data=cohort_data_capped,
    total_by_dt=total_by_dt_capped,
    baseline_start=b2_start.strftime('%Y-%m-%d'),
    baseline_end=b2_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b2_start.strftime("%b %d %Y")} Baseline 2',
)

In [297]:
def chart_relative_drop(cohort_data, total_by_dt, baseline_start, baseline_end, recent_start, recent_end, label):
    """Two % charts: cohort-level % change bar chart, and total DAU indexed to day 0 = 100.
    The indexed overlay y-axis covers the full extent of both series so no data falls off-chart."""

    # --- Cohort % change bar chart ---
    baseline = (
        cohort_data[cohort_data['dt'].between(baseline_start, baseline_end)]
        .groupby('install_dt_bin', observed=True)['users']
        .mean()
        .rename('baseline_avg_dau')
    )
    recent = (
        cohort_data[cohort_data['dt'].between(recent_start, recent_end)]
        .groupby('install_dt_bin', observed=True)['users']
        .mean()
        .rename('recent_avg_dau')
    )
    cohort_drop = pd.concat([baseline, recent], axis=1).fillna(0)
    cohort_drop['delta'] = cohort_drop['recent_avg_dau'] - cohort_drop['baseline_avg_dau']
    cohort_drop['pct_change'] = cohort_drop.apply(
        lambda r: (r['delta'] / r['baseline_avg_dau'] * 100) if r['baseline_avg_dau'] != 0 else float('nan'),
        axis=1,
    )
    cohort_drop = cohort_drop.dropna(subset=['pct_change']).sort_values('pct_change')

    #colors = ['#d62728' if v < 0 else '#2ca02c' for v in cohort_drop['pct_change']]
    #fig1 = go.Figure(go.Bar(
        #x=cohort_drop.index,
        #y=cohort_drop['pct_change'],
        #marker_color=colors,
        #text=[f"{v:+.1f}%" for v in cohort_drop['pct_change']],
        #textposition='outside',
        #customdata=list(zip(cohort_drop['baseline_avg_dau'].round(0), cohort_drop['recent_avg_dau'].round(0))),
        #hovertemplate=(
            #'<b>%{x}</b><br>'
            #'Baseline avg DAU: %{customdata[0]:,.0f}<br>'
            #'Recent avg DAU: %{customdata[1]:,.0f}<br>'
            #'Change: %{y:+.1f}%'
            #'<extra></extra>'
        #),
    #))
    #fig1.update_layout(
        #title=f"[{label}] % Change in Avg DAU by Cohort",
        #xaxis_title='Install Cohort',
        #yaxis=dict(title='% Change', ticksuffix='%'),
        #height=450, width=1100,
    #)
    #fig1.show()

    # --- Normalized overlay: total DAU indexed to day 0 = 100 ---
    baseline_daily = total_by_dt[total_by_dt['dt'].between(baseline_start, baseline_end)].sort_values('dt').reset_index(drop=True)
    recent_daily   = total_by_dt[total_by_dt['dt'].between(recent_start, recent_end)].sort_values('dt').reset_index(drop=True)
    baseline_daily['day'] = range(len(baseline_daily))
    recent_daily['day']   = range(len(recent_daily))
    baseline_daily['date_str'] = baseline_daily['dt'].dt.strftime('%Y-%m-%d')
    baseline_daily['dow']      = baseline_daily['dt'].dt.strftime('%A')
    recent_daily['date_str']   = recent_daily['dt'].dt.strftime('%Y-%m-%d')
    recent_daily['dow']        = recent_daily['dt'].dt.strftime('%A')

    baseline_idx = baseline_daily['users'] / baseline_daily['users'].iloc[0] * 100
    recent_idx   = recent_daily['users']   / recent_daily['users'].iloc[0]   * 100

    def regression_line(x, y):
        m, b = np.polyfit(x, y, 1)
        return m * x + b

    baseline_trend = regression_line(baseline_daily['day'].values, baseline_idx.values)
    recent_trend   = regression_line(recent_daily['day'].values,   recent_idx.values)

    # y-axis range: cover the full extent of both series (including trend lines) with 20% padding
    all_vals = pd.concat([baseline_idx, recent_idx,
                          pd.Series(baseline_trend), pd.Series(recent_trend)])
    y_pad   = (all_vals.max() - all_vals.min()) * 0.2
    y_range = [all_vals.min() - y_pad, all_vals.max() + y_pad]

    hover_tpl = (
        'Day %{x}<br>'
        'Date: %{customdata[0]}<br>'
        '%{customdata[1]}<br>'
        'Indexed DAU: %{y:.1f}'
        '<extra></extra>'
    )

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(
        x=baseline_daily['day'], y=baseline_idx,
        mode='lines+markers', name=f'Baseline ({baseline_start} → {baseline_end})',
        line=dict(color='steelblue'),
        customdata=baseline_daily[['date_str', 'dow']].values,
        hovertemplate=hover_tpl,
    ))
    fig2.add_trace(go.Scatter(
        x=baseline_daily['day'], y=baseline_trend,
        mode='lines', name='Baseline trend',
        line=dict(color='steelblue', dash='dash', width=2),
        hoverinfo='skip',
    ))
    fig2.add_trace(go.Scatter(
        x=recent_daily['day'], y=recent_idx,
        mode='lines+markers', name=f'Recent ({recent_start} → {recent_end})',
        line=dict(color='crimson'),
        customdata=recent_daily[['date_str', 'dow']].values,
        hovertemplate=hover_tpl,
    ))
    fig2.add_trace(go.Scatter(
        x=recent_daily['day'], y=recent_trend,
        mode='lines', name='Recent trend',
        line=dict(color='crimson', dash='dash', width=2),
        hoverinfo='skip',
    ))
    fig2.add_hline(y=100, line_dash='dot', line_color='grey', opacity=0.5)
    fig2.update_layout(
        title=f"[{label}] Total DAU Indexed to Day 0 = 100 (baseline vs recent)",
        xaxis_title='Day in period (0 = first day)',
        yaxis=dict(title='DAU indexed (day 0 = 100)', range=y_range),
        height=450, width=1100,
        legend=dict(orientation='h', yanchor='bottom', y=-0.25, xanchor='center', x=0.5),
    )
    fig2.show()

In [298]:
chart_relative_drop(
    cohort_data=cohort_data_capped,
    total_by_dt=total_by_dt_capped,
    baseline_start=b1_start.strftime('%Y-%m-%d'),
    baseline_end=b1_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b1_start.strftime("%b %d %Y")} Baseline 1',
)

chart_relative_drop(
    cohort_data=cohort_data_capped,
    total_by_dt=total_by_dt_capped,
    baseline_start=b2_start.strftime('%Y-%m-%d'),
    baseline_end=b2_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b2_start.strftime("%b %d %Y")} Baseline 2',
)

### 5.a DAU Drop Attribution — before 2025 Cohorts Only

Same analysis as section 5 but restricted to specific cohorts, to isolate how the oldest retained users are trending across the same comparison periods.

In [299]:
# Filter to 2021 & 2022 cohorts only, then restore globals after
cohort_data_filtered = pd.DataFrame()
total_by_dt_filtered = pd.DataFrame()

cohort_data_filtered = cohort_data_capped[cohort_data_capped['install_dt_bin'].isin(['2021', '2022', '2023', '2024'])]
total_by_dt_filtered = cohort_data_filtered.groupby('dt', observed=True)['users'].sum().reset_index()

analyze_dau_drop(
    cohort_data=cohort_data_filtered,
    total_by_dt=total_by_dt_filtered,
    baseline_start=b1_start.strftime('%Y-%m-%d'),
    baseline_end=b1_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b1_start.strftime("%b %d %Y")} Baseline 1 — 21,22,23,24 cohorts',
)

analyze_dau_drop(
    cohort_data=cohort_data_filtered,
    total_by_dt=total_by_dt_filtered,
    baseline_start=b2_start.strftime('%Y-%m-%d'),
    baseline_end=b2_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b2_start.strftime("%b %d %Y")} Baseline 2 — 21,22,23,24 cohorts',
)


In [300]:
chart_relative_drop(
    cohort_data=cohort_data_filtered,
    total_by_dt=total_by_dt_filtered,
    baseline_start=b1_start.strftime('%Y-%m-%d'),
    baseline_end=b1_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b1_start.strftime("%b %d %Y")} Baseline 1',
)

chart_relative_drop(
    cohort_data=cohort_data_filtered,
    total_by_dt=total_by_dt_filtered,
    baseline_start=b2_start.strftime('%Y-%m-%d'),
    baseline_end=b2_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b2_start.strftime("%b %d %Y")} Baseline 2',
)

### 5.b DAU Drop Attribution — after 2024 Cohorts Only

In [301]:
cohort_data_filtered = pd.DataFrame()
total_by_dt_filtered = pd.DataFrame()

cohort_data_filtered = cohort_data_capped[~cohort_data_capped['install_dt_bin'].isin(['2021','2022','2023','2024'])]
total_by_dt_filtered = cohort_data_filtered.groupby('dt', observed=True)['users'].sum().reset_index()

analyze_dau_drop(
    cohort_data=cohort_data_filtered,
    total_by_dt=total_by_dt_filtered,
    baseline_start=b1_start.strftime('%Y-%m-%d'),
    baseline_end=b1_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b1_start.strftime("%b %d %Y")} Baseline 1 — 25,26 cohorts',
)

analyze_dau_drop(
    cohort_data=cohort_data_filtered,
    total_by_dt=total_by_dt_filtered,
    baseline_start=b2_start.strftime('%Y-%m-%d'),
    baseline_end=b2_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b2_start.strftime("%b %d %Y")} Baseline 2 — 25,26 cohorts',
)


In [302]:
chart_relative_drop(
    cohort_data=cohort_data_filtered,
    total_by_dt=total_by_dt_filtered,
    baseline_start=b1_start.strftime('%Y-%m-%d'),
    baseline_end=b1_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b1_start.strftime("%b %d %Y")} Baseline 1',
)

chart_relative_drop(
    cohort_data=cohort_data_filtered,
    total_by_dt=total_by_dt_filtered,
    baseline_start=b2_start.strftime('%Y-%m-%d'),
    baseline_end=b2_end.strftime('%Y-%m-%d'),
    recent_start=recent_start.strftime('%Y-%m-%d'),
    recent_end=recent_end.strftime('%Y-%m-%d'),
    label=f'{b2_start.strftime("%b %d %Y")} Baseline 2',
)

## 6.Player Level analysis

In [266]:
# hide-output
pl_data_agg_lvl = pl_data.groupby(['max_level'], observed=True).agg(
    users=('user_id', 'nunique')
).reset_index()
pl_data_agg_lvl

,max_level,users
0,1,120440
1,2,164823
2,3,292802
3,4,302994
4,5,319438
...,...,...
195,196,11
196,197,8
197,198,9
198,199,11


In [267]:
# hide-output
pl_data['max_level_seg'] = pd.cut(pl_data['max_level'], bins=[0, 25, 50, 75, 100, 150, float('inf')], labels=['A.0-25', 'B.26-50', 'C.51-75', 'D.76-100', 'E.101-150', 'F.151+'])
pl_data['null_flag'] = pl_data['max_level'].isnull()

pl_data_agg_null = pl_data.groupby(['dt', 'platform','null_flag'], observed=True).agg(
    users=('user_id', 'nunique')
).reset_index()
pl_data_agg_null

,dt,platform,null_flag,users
0,2025-02-23,AND,False,76578
1,2025-02-23,IOS,False,110693
2,2025-02-24,AND,False,75569
3,2025-02-24,IOS,False,109840
4,2025-02-25,AND,False,75036
...,...,...,...,...
798,2026-03-27,IOS,False,55498
799,2026-03-28,AND,False,39294
800,2026-03-28,IOS,False,54801
801,2026-03-29,AND,False,40220


In [268]:
# hide-output
pl_data_agg_null['total_users'] = pl_data_agg_null.groupby(['dt', 'platform'])['users'].transform('sum')
pl_data_agg_null['pct_of_platform'] = (pl_data_agg_null['users'] / pl_data_agg_null['total_users'] * 100).round(2)
pl_data_agg_null

,dt,platform,null_flag,users,total_users,pct_of_platform
0,2025-02-23,AND,False,76578,76578,100.0
1,2025-02-23,IOS,False,110693,110693,100.0
2,2025-02-24,AND,False,75569,75569,100.0
3,2025-02-24,IOS,False,109840,109840,100.0
4,2025-02-25,AND,False,75036,75036,100.0
...,...,...,...,...,...,...
798,2026-03-27,IOS,False,55498,55498,100.0
799,2026-03-28,AND,False,39294,39294,100.0
800,2026-03-28,IOS,False,54801,54801,100.0
801,2026-03-29,AND,False,40220,40220,100.0


## Output file

In [270]:
export_notebook_html(
    notebook_path='./exploration.ipynb',
    output_path='./notebook_output.html',
)

Saved to notebook_output.html


PosixPath('notebook_output.html')